# Generate Figure log($\textrm{el}_1/\textrm{el}_2$) vs. $V_{\textrm{tan}}$

This is mainly to create a figure of log(Li/Ca) vs. Vtan, but I figure I might as well code it semi-generically while I'm at it to see if there's another apparent trend in the data

I'll also need to retrieve the full Gaia solutions for all of the relevant objects, but I think I'll just separately calculate the V_tan values and then put them into the same "abundances" file that I use for white dwarf abundances and ages and just read them in from there to get them in here for plotting. Make my life easier.

In [1]:
from __future__ import print_function

import matplotlib

matplotlib.use('pdf')
savefig=True
    
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coords
from astropy import units as u
from astropy import constants as const
from astropy import convolution as conv
from astropy.table import Table, Column
import scipy.interpolate as scinterp
import time
import periodictable as pt

start = time.time()
print(start)
time_string=str(start).split('.')[0]

from mendeleev import O, Ca, Li, Na, Si, Fe, Mg, He
start = time.time()

#import wdatmos
import spec_plot_tools as spt
import cal_params as cp
import plot_spec as ps
import abundance_corrections as acorr
import interp_tau as itau
import fix_strings as fs


#print(os.getcwd())

1612201961.729331


No handles with labels found to put in legend.


all_avg
(116, 4, 27)
(4, 27, 116)
(116,)
(116,)
(27, 116)
(116,)
(116,)
(27, 116)
(116,)
(116,)
(27, 116)
(116,)
(116,)
(27, 116)


In [2]:
plt.show()

In [3]:
figure_output_dir='/Users/BenKaiser/Desktop/'
target_dir= '/Users/BenKaiser/Desktop/radial_velocity_calculations/'
os.chdir(target_dir)

In [4]:
#wd_abund_file='all_wd_abundances.csv'
wd_abund_file='20210131_all_wd_abundances.csv'
lodders_abund_file='Lodders2009_solarsystem_abundances.csv'
solar_system_object_file='solar_system_body_abundances.csv'

In [5]:
wd_abund_table=Table.read(wd_abund_file)
lodders_table=Table.read(lodders_abund_file)
bodies_table=Table.read(solar_system_object_file)
lodders_table.add_index('element')
wd_abund_table.add_index('name')
bodies_table.add_index('name')

limit_length=0.3 #length of limit error bars on plots
limit_indicator=99. #value above which if the absolute value of the error on a measurement is above it indicates it should be a limit



In [6]:
use_indices=np.where(bodies_table['show']==1)
use_bodies_table=bodies_table[use_indices]

In [7]:
color_dict={
    'WDJ1644-0449':'#ff0000',
    'SDSSJ1330+6435':'#8900ff',
    'WDJ2356-209':'#00ffc5',
    'SDSSJ1636+1619':'pink',
    'WDJ2317+1830':'orange',
    'WDJ1824+1213':'g',
    'LHS2534':'b'
}
step_dict={
    'WDJ1644-0449':5,
    'SDSSJ1330+6435':5,
    'WDJ2356-209':5,
    'SDSSJ1636+1619':5,
    'WDJ2317+1830':5,
    'WDJ1824+1213':5,
    'LHS2534':5
}

wd_marker='*'
#met_marker='D'
#ssp_marker='met_marker'
met_marker='s'
ssp_marker='D'
met_color='#1ca1f2'
met_size=3
ci_size=6
wd_size=10
starsize=4
dp_alpha=0.5
ci_leg_size=9
arr_naca=[-0.4,-0.1]
alpha_range=[0.5,0.2]
arrow_segs=100
arrow_width=0.03
#arrow_width=0.07
legend_font=7

arrow_line=4
figure_text_size=6
default_offset=[0.05,0.00]
annot_line_weight=0.03

star_marker='o'
pop_colors=['darkorange','brown','navy','grey'] #thin disk, thick disk, halo, in-between for Bensby plots
#pop_colors=['darkorange','darkorange','darkorange','darkorange'] #for version where we don't distinguish pops




In [8]:
def plot_wd_el1el2vtan_errorbar(vtan_lsr, el1el2, vtan_lsr_err, el1el2_err, name, selected_marker=wd_marker, markersize=wd_size, label=''):
    if label=='':
        label=name
    else:
        pass
    uplims=False
    lolims=False
    xlolims=False
    xuplims=False
    
    if np.abs(el1el2_err)> limit_indicator:
        if el1el2_err > 0:
            lolims=True
        elif el1el2_err < 0:
            uplims=True
        else:
            print("This shouldn't print el1el2_err")
        el1el2_err= limit_length
    else:
        #no limit indicators are present for the 2 relative abundances input
        pass
    plt.errorbar(vtan_lsr, el1el2, yerr= el1el2_err, xerr= vtan_lsr_err, uplims=uplims, lolims=lolims, xuplims=xuplims, xlolims=xlolims, color=color_dict[name],marker=selected_marker,  markersize=markersize,linestyle='None')
    plt.errorbar(vtan_lsr,el1el2, label=fs.fix_display_string(label), marker=selected_marker, markersize=markersize, color=color_dict[name],linestyle='None')
    return

In [9]:

def plot_lica_vtan(row,sol_norm=True,lica_string='li/ca', color=None, use_central_vals=True,SSP=True,t_step_units='Ca',add_arrow=True,t_step=5,t_max=5):
    print('\n\n',row['name'],'\n\n')
    vtan_lsr_err= [[row['vtan_lsr_err_lo']],[row['vtan_lsr_err_hi']]]
    #age_errs= row['age_plus'],row['age_minus']
    #age_errs=[row['age_minus'],row['age_plus']]
    lica_err= row[lica_string+'_err']
    target_row=row
    target_lica=row[lica_string]
    string1= lica_string
    string2="na/ca" #we need a placeholder for the declining phase function for a second abundance ratio even though we won't use it
    #print('age_errs',age_errs.shape)
    #plt.errorbar(row['age'],row[lica_string],xerr=row['age_minus'],label=row['name'],marker='o')
    if SSP:
        target_lica, target_kca, lica_err, kca_err=acorr.easy_dist_ssp(target_row,['Li','Ca','Na'], plot_all=False,tau_rand=True)
        plot_wd_el1el2vtan_errorbar(row['vtan_lsr'], target_lica, vtan_lsr_err, lica_err, row['name'], selected_marker=ssp_marker, markersize=ci_size, label=row['name']+' SSP')
        if add_arrow:
            tau_time= 10.**itau.extrapolate_tau_x_logg(row['teff'], row['logg'], t_step_units)
            tau_time=tau_time*1e-6 #converted to Myr
            t_step=t_step*tau_time
            t_max=t_max*tau_time
            times=t_step
            arrow_endy,throwaway=acorr.el1el2_DP_el3el2_ftimes(row['teff'], row[string1], row[string2],times, 'Li', 'Ca', 'Na', logg=row['logg'])
            arrow_endx=float(row['vtan_lsr'])
            #print('arrow_endy',arrow_endy,'arrow_endx',arrow_endx)
            #plt.plot(arrow_endx,arrow_endy,marker='o')
            ypoints=np.linspace(target_lica,arrow_endy,arrow_segs)
            xpoints=np.linspace(float(row['vtan_lsr']),arrow_endx,arrow_segs)
            #dx=arrow_endx-target_el3el2
            #dy=arrow_endy-target_el1el2
            dx=arrow_endx-xpoints[-2]
            dy=arrow_endy-ypoints[-2]
            def get_segs(points):
                return np.vstack([points,np.roll(points,1)]).T[1:]
            x_segs=get_segs(xpoints)
            y_segs=get_segs(ypoints)
            #print('x_segs',x_segs)
            #color_array=np.empty_like(ypoints,dtype=str)
            #color_array[:]=color_dict[name]
            alpha_vals=np.linspace(alpha_range[0],alpha_range[1],arrow_segs)
            #print('alpha_vals',alpha_vals)
            #for x,y,alpha in zip(x_segs, y_segs,alpha_vals):
                #plt.plot(x,y,color=color_dict[name],alpha=alpha,linewidth=arrow_line)
                #plt.plot(x,y,color=color_dict[row['name']],alpha=alpha,linewidth=arrow_line)
                #plt.plot(x,y,color=color_dict[row['name']],alpha=alpha,linewidth=arrow_line)
            #plt.arrow(xpoints[-2],ypoints[-2],dx,dy,color=color_dict[row['name']],width=arrow_width, alpha=alpha_range[1],length_includes_head=True )
            plt.arrow(row['vtan_lsr'],target_lica,arrow_endx-float(row['vtan_lsr']),arrow_endy-target_lica,color=color_dict[row['name']],width=arrow_width,alpha=alpha_range[0],length_includes_head=True)
        else:
            pass
            plt.ylabel(r'log(Li/Ca)')
    elif use_central_vals:
        plot_wd_el1el2vtan_errorbar(row['vtan_lsr'], target_lica, vtan_lsr_err, lica_err, row['name'], selected_marker=wd_marker, markersize=wd_size, label='')
    else:
        #doing upper limits only because we don't know the phase.
        print("doing upper limits exclusively... shouldn't have happened")
        lica_val=row[lica_string]+3*lica_err #3-sigma upper side of the distribution
        plt.errorbar(float(row['age']),lica_val,marker=wd_marker, label=row['name'],color=color_dict[row['name']],markersize=wd_size, linestyle='None')
        plt.errorbar(float(row['age']),lica_val,xerr=age_errs,yerr=0.3,marker=wd_marker,uplims=True,color=color_dict[row['name']],markersize=wd_size, linestyle='None')
        plt.ylabel(r'$\log$(Li/Ca)')
    return

In [10]:
#plt.figure(figsize=(15,10))
spt.initiate_science_plot()
#plt.figure(figsize=(4.75,4.75),constrained_layout=True)
plt.figure(figsize=(7.25,7.25),constrained_layout=False)




count=0
#bp.plot_lica_age_pop(colors=pop_colors, rep_errors=True, sol_norm=False,markersize=starsize)
plt.errorbar(0., 1.10-6.33, yerr=np.sqrt(0.1**2+0.07**2),marker=star_marker, color=met_color, label="Sun",linestyle='None',markersize=starsize )
plt.errorbar(0,lodders_table.loc['Li']['A_el']-lodders_table.loc['Ca']['A_el'],yerr=np.sqrt(lodders_table.loc['Li']['A_el_err']**2+lodders_table.loc['Ca']['A_el_err']**2),label="CI Chondrites",marker=met_marker, color=met_color, linestyle='None', markersize=ci_leg_size)
plt.errorbar(0,lodders_table.loc['Li']['A_el']-lodders_table.loc['Ca']['A_el'],yerr=np.sqrt(lodders_table.loc['Li']['A_el_err']**2+lodders_table.loc['Ca']['A_el_err']**2),marker=met_marker,markersize=ci_size, color=met_color)
#plt.xlim(14,0)

for row in wd_abund_table:
    #plot_lica_age(row,sol_norm=False,lica_string='li/ca', use_central_vals=False)
    plot_lica_vtan(row,sol_norm=False,lica_string='li/ca', use_central_vals=True,SSP=False)
    plot_lica_vtan(row,sol_norm=False,lica_string='li/ca', use_central_vals=True,SSP=True)
    #if row['name']=='WDJ2356-209':
        
    count+=1

#plt.xlim(15,0)


#plt.axvline(x=13.8, linestyle=':', color='k')
plt.legend(fontsize=legend_font)
#plt.grid(True)
#plt.xlabel(r'$V_{\textrm{tan LSR}}$ (km/s)')
plt.ylabel('log(Li/Ca)')
plt.xlabel('V_tanLSR (km/s)')



if savefig:
    print(os.getcwd())
    os.chdir(figure_output_dir)
    print(os.getcwd())
    start = time.time()
    print(start)
    time_string=str(start).split('.')[0]
    plt.savefig('lica_vtan_'+time_string+'.pdf')#plt.grid(True)
else:
    pass
plt.show()



 WDJ1644-0449 


bad_string: GaiaJ1644-0449
input_string: WDJ1644-0449
output_string: WDJ1644-0449
bad_string: SDSSJ1330+6435
input_string: WDJ1644-0449
output_string: WDJ1644-0449
bad_string: WDJ2356-209
input_string: WDJ1644-0449
output_string: WDJ1644-0449
bad_string: Gaia J1644-0449
input_string: WDJ1644-0449
output_string: WDJ1644-0449
bad_string: WD J2356-209
input_string: WDJ1644-0449
output_string: WDJ1644-0449


 WDJ1644-0449 


3884.809442932467 7.640484496229651
3884.809442932467 7.640484496229651
3884.809442932467 7.640484496229651
tau_Li-tau_Ca 0.5520497476145002 +/- 0.21561543212374978
tau_Na-tau_Ca 0.34844189066944337 +/- 0.2076824564447466
target_ssp Li Ca -2.2587337176560647
target_ssp Na Ca -0.27500746704702733
dist ssp Li Ca -2.256796113595304 -2.2518825454735585 0.2826302940929403
dist ssp Na Ca -0.26867916630504574 -0.2705233137764866 0.2694283002566265
bad_string: GaiaJ1644-0449
input_string: WDJ1644-0449 SSP
output_string: WDJ1644-0449 SSP
bad_string: SDSSJ1330

tau_Li-tau_Ca 0.5612200841735846 +/- 0.20076259992219783
tau_Na-tau_Ca 0.3565271380708061 +/- 0.1997265457883249
target_ssp Li Ca -2.3120148849056044
target_ssp Na Ca 0.18915485156247347
dist ssp Li Ca -2.311641826923002 -2.3110662179730777 0.22296239785197022
dist ssp Na Ca 0.19363202262281037 0.19245296909101814 0.23393270319818568
bad_string: GaiaJ1644-0449
input_string: LHS2534 SSP
output_string: LHS2534 SSP
bad_string: SDSSJ1330+6435
input_string: LHS2534 SSP
output_string: LHS2534 SSP
bad_string: WDJ2356-209
input_string: LHS2534 SSP
output_string: LHS2534 SSP
bad_string: Gaia J1644-0449
input_string: LHS2534 SSP
output_string: LHS2534 SSP
bad_string: WD J2356-209
input_string: LHS2534 SSP
output_string: LHS2534 SSP
Li log tau 6.518049922463561
Ca log tau 5.956035037557957
Na log tau 6.316880185995483
/Users/BenKaiser/Desktop/radial_velocity_calculations
/Users/BenKaiser/Desktop
1612201997.2676659
